# PDF → YOLO Dataset Generation (page-level / tiled / horizontal strips)

End-to-end, **run-all** pipeline that turns the `processed/` folder of annotated
scanned PDFs (each with a sidecar `result(s).json` of highlight boxes) into YOLO
datasets and zips them into `/kaggle/working/`:

1. **`dataset_page_level.zip`** — full pages (rendered @300 DPI, no deskew).
2. **`dataset_strips.zip`** — each page split into **N full-width horizontal
   strips** with an adaptive vertical overlap, so **no highlight box is ever
   cut** (highlights are wide, short horizontal lines → full-width strips never
   cut them in x; the overlap guarantees they are never cut in y either).
3. **`dataset_tiled.zip`** — square overlapping tiles via
   `ultralytics.data.split_dota` (kept as an alternative).

👉 Use the **`BUILD_DATASETS`** flag in the Configuration cell to choose which
datasets to build & zip: any combination of `"page"`, `"strips"`, `"tiled"`.
Default: `["page", "strips"]`.

> All logic is taken verbatim from the tested `build_holo_datasets.py`.
> This notebook just drops the `argparse` CLI (so run-all works) and auto-detects
> the input `processed/` directory under `/kaggle/input/`.

**Why horizontal strips?** With this data, square tiles (`crop_size=1024`) drop
~13.5% of the widest highlights because of the hardcoded `iof_thr=0.7` in
`split_dota`. Full-width horizontal strips never cut a box in x, and an
**adaptive overlap** (≥ 1.5× the tallest box on each page) guarantees every box
is fully inside at least one strip — so nothing is lost and the page can be
reconstructed from the strip names (`{stem}__strip{i}__y{y0}__h{sh}.jpg`).

**How many strips?** `N_STRIPS=4` (default) balances speed and precision: each
A4 strip is ~880 px tall (full width 2480 px), 4× more training images than full
pages, and every box (max height 0.217×page < 1/4 page) fits entirely in one
strip. Use `N_STRIPS=3` for fewer/larger strips (faster, fewer boundary overlaps).


## 0 · Install dependencies (idempotent)

In [1]:
import importlib, subprocess, sys

for pip_name, mod in [("pymupdf", "fitz"),
                      ("opencv-python", "cv2"),
                      ("ultralytics", "ultralytics"),
                      ("shapely", "shapely")]:
    try:
        importlib.import_module(mod)
        print(f"ok: {mod}")
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


installing pymupdf ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 73.3 MB/s eta 0:00:00
ok: cv2
installing ultralytics ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.2 MB/s eta 0:00:00
ok: shapely


## 1 · Configuration

Auto-detects the `processed/` directory under `/kaggle/input/`.
Override `INPUT_DIR` manually if your dataset is mounted elsewhere.

**`BUILD_DATASETS`** selects which datasets to build & zip. The page-level
dataset is always built as the source for `strips`/`tiled`; it is zipped only if
`"page"` is in the list.

The split is **70/15/15 with PDF-level no-leakage** (seed 42); negatives are
balanced to ~12% (10-15% band) at page level.

In [2]:
import os, glob

# ── Which datasets to build & zip ──────────────────────────────────────────
# Any combination of: "page", "strips", "tiled".  Default: page + strips.
BUILD_DATASETS = ["page"]   # e.g. ["strips"], ["page","tiled","strips"], ...

# ── Auto-detect input ──────────────────────────────────────────────────────
INPUT_DIR = os.environ.get("INPUT_DIR", "")

if not INPUT_DIR:
    candidates = []
    for base in glob.glob("/kaggle/input/*"):
        # dataset root containing 'processed/'
        if os.path.isdir(os.path.join(base, "processed")):
            candidates.append(base)
        # the dir itself is the processed folder (has subfolders with PDFs)
        if any(glob.glob(os.path.join(base, "*", "*.pdf"))):
            candidates.append(base)
    candidates = sorted(set(candidates))
    if candidates:
        INPUT_DIR = candidates[0]
        print("Auto-detected INPUT_DIR =", INPUT_DIR)
        if len(candidates) > 1:
            print("Other candidates:", candidates[1:])
    else:
        # local fallback (for testing outside Kaggle)
        for fb in ("./processed", "../processed", "/kaggle/input/datasets/frabbate/holo-real-clean/processed"):
            if os.path.isdir(fb):
                INPUT_DIR = os.path.abspath(fb)
                print("Fallback INPUT_DIR =", INPUT_DIR)
                break

if not INPUT_DIR or not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(
        "Could not auto-detect a 'processed/' directory under /kaggle/input/. "
        "Set INPUT_DIR manually in this cell."
    )

# ── Pipeline parameters ────────────────────────────────────────────────────
OUTPUT_DIR  = "/kaggle/working"     # where the .zip archives are written
SEED        = 42
DPI         = 300                   # render DPI (zoom = DPI/72)
NEG_PCT     = 0.12                  # target negative-page ratio (10-15% band)

# Strips dataset
N_STRIPS         = 4                # horizontal strips per page (3 or 4 recommended)
STRIP_GAP_FRAC   = 0.15            # min overlap as fraction of strip height
                                  # (enlarged per-page to >=1.5x tallest box -> no box cut)

# Tiled dataset (square crops, alternative)
CROP_SIZE   = 2048                 # spec example 1024 drops ~13.5% of wide highlights
GAP         = 200                  # tiling overlap

MAX_DIM     = 4096                 # cap rendered long side in px (0 = pure 300 DPI)
KEEP_DIRS   = False                # keep unzipped dirs after zipping?

print("INPUT_DIR  =", INPUT_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("BUILD_DATASETS =", BUILD_DATASETS)
print("params: seed=%d dpi=%d neg_pct=%.2f n_strips=%d strip_gap=%.2f crop=%d gap=%d max_dim=%d keep_dirs=%s"
      % (SEED, DPI, NEG_PCT, N_STRIPS, STRIP_GAP_FRAC, CROP_SIZE, GAP, MAX_DIM, KEEP_DIRS))


Fallback INPUT_DIR = /kaggle/input/datasets/frabbate/holo-real-clean/processed
INPUT_DIR  = /kaggle/input/datasets/frabbate/holo-real-clean/processed
OUTPUT_DIR = /kaggle/working
BUILD_DATASETS = ['page']
params: seed=42 dpi=300 neg_pct=0.12 n_strips=4 strip_gap=0.15 crop=2048 gap=200 max_dim=4096 keep_dirs=False


## 2 · Imports, constants & logging

In [3]:
from __future__ import annotations

import argparse
import glob
import json
import logging
import os
import random
import re
import shutil
import sys
import traceback
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

# --------------------------------------------------------------------------- #
# Constants
# --------------------------------------------------------------------------- #
DATA_YAML = (
    "path: /kaggle/working/dataset\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "nc: 1\n"
    "names:\n"
    "  0: highlight\n"
)

SPLITS = ("train", "val", "test")
CLASS_ID = 0  # single class: "highlight"


# --------------------------------------------------------------------------- #
# Logging
# --------------------------------------------------------------------------- #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("holo")


## 3 · Dependency bootstrap & filesystem helpers

In [4]:
def ensure_deps() -> None:
    """pip-install missing runtime deps (no-op if already present)."""
    import importlib
    import subprocess

    requirements = {
        "pymupdf": "fitz",
        "opencv-python": "cv2",
        "ultralytics": "ultralytics",
        "shapely": "shapely",
        "pyyaml": "yaml",
    }
    for pip_name, mod in requirements.items():
        try:
            importlib.import_module(mod)
        except ImportError:
            log.info("Installing missing dependency: %s", pip_name)
            try:
                subprocess.check_call(
                    [sys.executable, "-m", "pip", "install", "-q", pip_name]
                )
            except Exception as e:  # pragma: no cover
                log.warning("Could not install %s: %s", pip_name, e)


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def sanitize_stem(name: str, max_len: int = 80) -> str:
    """Make a filesystem/glob-safe stem from an arbitrary PDF basename."""
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("._-")
    s = re.sub(r"_+", "_", s)
    if not s:
        s = "doc"
    return s[:max_len]


def _safe_move(src: str, dst: str) -> None:
    """Move src->dst, overwriting dst if it already exists (same filesystem)."""
    if os.path.exists(dst):
        os.remove(dst)
    shutil.move(src, dst)


def _write_yaml(dataset_dir: str, content: str = DATA_YAML) -> None:
    with open(os.path.join(dataset_dir, "data.yaml"), "w", encoding="utf-8") as f:
        f.write(content)


def _link_or_copy(src: str, dst: str) -> None:
    """Symlink src->dst, falling back to hardlink then copy."""
    if os.path.exists(dst) or os.path.islink(dst):
        return
    try:
        os.symlink(os.path.abspath(src), dst)
        return
    except OSError:
        pass
    try:
        os.link(os.path.abspath(src), dst)
        return
    except OSError:
        pass
    shutil.copy2(src, dst)


def _fitz_page_count(pdf_path: str) -> int:
    import fitz
    doc = fitz.open(pdf_path)
    n = doc.page_count
    doc.close()
    return n


## 4 · Phase 1 — Find PDF / result.json pairs

In [5]:
def find_pdf_json_pairs(input_dir: str) -> List[Tuple[str, str]]:
    """Recursively scan 'processed/' (or input_dir itself) and return
    a list of (pdf_path, json_path) tuples. PDFs without a sidecar
    result.json / results.json are skipped with a warning."""
    processed = os.path.join(input_dir, "processed")
    if not os.path.isdir(processed):
        processed = input_dir  # tolerate --input pointing straight at processed/
    if not os.path.isdir(processed):
        raise FileNotFoundError(f"No 'processed/' directory under {input_dir!r}")

    pdf_files: List[str] = []
    for root, _dirs, files in os.walk(processed):
        for fn in files:
            if fn.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(root, fn))
    pdf_files.sort()

    pairs: List[Tuple[str, str]] = []
    for pdf in pdf_files:
        d = os.path.dirname(pdf)
        json_path = None
        for name in ("result.json", "results.json"):
            cand = os.path.join(d, name)
            if os.path.exists(cand):
                json_path = cand
                break
        if json_path is None:
            log.warning("No result.json/results.json for PDF -> skipping: %s", pdf)
            continue
        pairs.append((pdf, json_path))

    log.info(
        "Found %d PDF/JSON pair(s); skipped %d PDF(s) without annotations.",
        len(pairs), len(pdf_files) - len(pairs),
    )
    return pairs


## 5 · Phase 2 — Render PDF pages with PyMuPDF (300 DPI)

In [6]:
def render_pdf_pages(
    pdf_path: str,
    dpi: int = 300,
    max_dim: int = 0,
    out_dir: str = ".",
    stem: str = "page",
):
    """Render every page of *pdf_path* to PNG in *out_dir*.

    Naming: {stem}_page_{num:04d}.png  (num is 0-indexed, matching results.json keys).

    Args:
        pdf_path: PDF to render.
        dpi: rendering DPI (zoom = dpi/72).
        max_dim: if > 0, pages whose long side at *dpi* exceeds max_dim are
                 rendered with a reduced zoom so the long side == max_dim.
                 Normalised annotations stay valid. 0 disables capping.
        out_dir: destination directory.
        stem: unique stem prefix (e.g. "012_<sanitized_basename>").

    Returns:
        list of dicts: {page_num, img_path, width, height, zoom}
    """
    import fitz

    os.makedirs(out_dir, exist_ok=True)
    base_zoom = dpi / 72.0
    doc = fitz.open(pdf_path)
    rendered = []
    try:
        for i in range(doc.page_count):
            page = doc[i]
            zoom = base_zoom
            if max_dim and max_dim > 0:
                r = page.rect
                long_px = max(r.width, r.height) * base_zoom
                if long_px > max_dim:
                    zoom = base_zoom * (max_dim / long_px)
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat)
            fname = f"{stem}_page_{i:04d}.png"
            fpath = os.path.join(out_dir, fname)
            pix.save(fpath)
            rendered.append(
                dict(page_num=i, img_path=fpath, width=pix.width, height=pix.height, zoom=zoom)
            )
    finally:
        doc.close()
    return rendered


## 6 · Phase 3 — No deskew (disabled)

In [7]:
def deskew_image(img_array: np.ndarray):
    """No-op kept for compatibility: returns the image unchanged and angle 0."""
    return img_array, 0.0


def deskew_image_file(img_path: str) -> float:
    """No-op kept for compatibility: leaves the rendered page unchanged."""
    return 0.0


## 7 · Phase 4 — Convert result.json → YOLO labels

In [8]:
def convert_annotations(json_path: str, img_size: Tuple[int, int] = None) -> Dict[str, List[List[float]]]:
    """Parse a result(s).json file into per-page YOLO lines.

    Returns:
        {page_key_str: [ [cls, cx, cy, w, h], ... ]}  (all values normalised 0-1).

    Handles the observed schema:
        {"pages": {"0": [ {"bbox": {x1,y1,x2,y2}, "bbox_yolo": {cx,cy,w,h} }, ... ]}}
    Coordinates are already normalised; *img_size* is only used to denormalise
    pixel-space fallback boxes if their values exceed 1.0.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    pages = data.get("pages", {})
    out: Dict[str, List[List[float]]] = {}

    # Support both dict ("0": [...]) and list ([ [...], ... ]) page containers.
    if isinstance(pages, dict):
        items = pages.items()
    elif isinstance(pages, list):
        items = enumerate(pages)
    else:
        items = []

    for page_key, anns in items:
        lines: List[List[float]] = []
        if not isinstance(anns, list):
            anns = []
        for a in anns:
            try:
                if isinstance(a, dict) and "bbox_yolo" in a:
                    b = a["bbox_yolo"]
                    cx, cy, w, h = (float(b["cx"]), float(b["cy"]), float(b["w"]), float(b["h"]))
                elif isinstance(a, dict) and "bbox" in a:
                    b = a["bbox"]
                    x1, y1, x2, y2 = (float(b["x1"]), float(b["y1"]), float(b["x2"]), float(b["y2"]))
                    if img_size and (x1 > 1.0 or y1 > 1.0 or x2 > 1.0 or y2 > 1.0):
                        iw, ih = img_size
                        x1, y1, x2, y2 = x1 / iw, y1 / ih, x2 / iw, y2 / ih
                    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                    w, h = x2 - x1, y2 - y1
                elif isinstance(a, (list, tuple)) and len(a) >= 4:
                    x1, y1, x2, y2 = (float(v) for v in a[:4])
                    if img_size and (x1 > 1.0 or y1 > 1.0 or x2 > 1.0 or y2 > 1.0):
                        iw, ih = img_size
                        x1, y1, x2, y2 = x1 / iw, y1 / ih, x2 / iw, y2 / ih
                    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                    w, h = x2 - x1, y2 - y1
                else:
                    continue
            except Exception as e:
                log.warning("Malformed annotation in %s: %s (%s)", json_path, a, e)
                continue

            cx = min(max(cx, 0.0), 1.0)
            cy = min(max(cy, 0.0), 1.0)
            w = min(max(w, 0.0), 1.0)
            h = min(max(h, 0.0), 1.0)
            if w <= 0.0 or h <= 0.0:
                continue
            lines.append([CLASS_ID, cx, cy, w, h])
        out[str(page_key)] = lines
    return out


## 8 · Phase 5 — Balance negative samples (10-15%)

In [9]:
def balance_negatives(samples: List[dict], target_pct: float = 0.12) -> List[dict]:
    """Ensure negatives are ~target_pct of the dataset.

    *samples* already contains every candidate page (positives + all natural
    negatives, i.e. pages with no objects). If negatives exceed the target they
    are randomly subsampled (seed 42). If they are below 10% a warning is logged
    (no positives are ever turned into negatives).

    Returns the selected sample list (positives + chosen negatives), shuffled.
    """
    positives = [s for s in samples if not s["is_negative"]]
    negatives = [s for s in samples if s["is_negative"]]

    if positives:
        desired = int(round(target_pct * len(positives) / (1.0 - target_pct)))
    else:
        desired = len(negatives)
    desired = max(desired, 0)

    rng = random.Random(42)
    if len(negatives) > desired:
        chosen = rng.sample(negatives, desired)
        log.info(
            "Negatives %d > desired %d -> subsampled to %d (target %.0f%%).",
            len(negatives), desired, desired, target_pct * 100,
        )
    else:
        chosen = list(negatives)
        total = len(positives) + len(chosen)
        actual = (len(chosen) / total) if total else 0.0
        if actual < 0.10:
            log.warning(
                "Only %d negatives available (%.1f%%) -> cannot reach 10%% target.",
                len(chosen), actual * 100,
            )
        else:
            log.info("Negatives %d <= desired %d -> kept all (%.1f%%).",
                     len(negatives), desired, actual * 100)

    selected = positives + chosen
    rng.shuffle(selected)

    total = len(selected)
    neg_pct = (len(chosen) / total) if total else 0.0
    log.info("Selected %d pages (positives=%d, negatives=%d, neg%%=%.1f%%).",
             total, len(positives), len(chosen), neg_pct * 100)
    return selected


## 9 · Phase 6 — 70/15/15 split with PDF-level no-leakage

In [10]:
def split_no_leakage(
    pdf_pages: List[dict], ratios: Tuple[float, float, float] = (0.7, 0.15, 0.15)
) -> Dict[str, List[dict]]:
    """Greedy PDF-level split. *pdf_pages* is a list of page-sample dicts,
    each carrying a 'pdf_key'. All pages of a given PDF go to exactly one split.

    Algorithm:
      - group pages by pdf_key
      - deterministic shuffle of pdf_keys (random.Random(42))
      - greedily fill train until 70% of total pages, then val until 85%, rest -> test
      - verify no PDF appears in more than one split
    """
    groups: Dict[str, List[dict]] = {}
    for s in pdf_pages:
        groups.setdefault(s["pdf_key"], []).append(s)
    pdf_keys = list(groups.keys())
    rng = random.Random(42)
    rng.shuffle(pdf_keys)

    total = sum(len(groups[k]) for k in pdf_keys)
    targets = [total * r for r in ratios]
    order = ["train", "val", "test"]

    splits: Dict[str, List[dict]] = {s: [] for s in SPLITS}
    counts = {s: 0 for s in SPLITS}
    idx = 0
    for k in pdf_keys:
        while idx < 2 and counts[order[idx]] >= targets[idx]:
            idx += 1
        sp = order[idx]
        splits[sp].extend(groups[k])
        counts[sp] += len(groups[k])

    # Post-split verification (no leakage)
    seen: Dict[str, str] = {}
    for sp in SPLITS:
        for s in splits[sp]:
            key = s["pdf_key"]
            if key in seen and seen[key] != sp:
                raise RuntimeError(f"No-leakage violation: PDF {key} in {seen[key]} and {sp}")
            seen[key] = sp

    log.info(
        "Split (pages): train=%d (%.1f%%) val=%d (%.1f%%) test=%d (%.1f%%) | total=%d",
        counts["train"], (counts["train"] / total * 100) if total else 0,
        counts["val"], (counts["val"] / total * 100) if total else 0,
        counts["test"], (counts["test"] / total * 100) if total else 0,
        total,
    )
    return splits


## 10 · Phase 7 — Build Dataset 1 (page-level) + data.yaml

In [11]:
def build_page_level_dataset(
    splits: Dict[str, List[dict]], out_dir: str, yaml_content: str = DATA_YAML
) -> Dict[str, int]:
    """Assemble Dataset 1: move rendered PNGs into images/{split}/, write
    YOLO .txt labels into labels/{split}/ (empty file for negatives), data.yaml.

    Returns: {split: n_pages}.
    """
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    for sp in SPLITS:
        os.makedirs(os.path.join(out_dir, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(out_dir, "labels", sp), exist_ok=True)

    counts: Dict[str, int] = {s: 0 for s in SPLITS}
    for sp in SPLITS:
        for s in splits[sp]:
            src_img = s.get("img_path")
            stem = s["stem"]
            if not src_img or not os.path.exists(src_img):
                log.warning("Missing rendered image for %s page %s -> skipping", s["pdf_key"], s["page_num"])
                continue
            dst_img = os.path.join(out_dir, "images", sp, stem + ".png")
            dst_lbl = os.path.join(out_dir, "labels", sp, stem + ".txt")
            _safe_move(src_img, dst_img)
            with open(dst_lbl, "w", encoding="utf-8") as f:
                for box in s.get("boxes", []):
                    f.write(f"{int(box[0])} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
            # no boxes -> 0-byte file (negative sample)
            counts[sp] += 1
    _write_yaml(out_dir, yaml_content)
    log.info("Dataset 1 (page-level) written to %s", out_dir)
    return counts


## 11 · Polygon ↔ YOLO helpers (for the tiled bridge)

In [12]:
def _yolo_box_to_polygon(cx: float, cy: float, w: float, h: float) -> List[float]:
    """[cls, x1,y1, x2,y1, x2,y2, x1,y2]  (axis-aligned rectangle as DOTA polygon)."""
    x1, y1 = cx - w / 2.0, cy - h / 2.0
    x2, y2 = cx + w / 2.0, cy + h / 2.0
    return [CLASS_ID, x1, y1, x2, y1, x2, y2, x1, y2]


def _yolo_label_file_to_polygon(lbl_path: str) -> List[List[float]]:
    polys: List[List[float]] = []
    if not os.path.exists(lbl_path):
        return polys
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) >= 5:
                c, cx, cy, w, h = (float(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4]))
                polys.append(_yolo_box_to_polygon(cx, cy, w, h))
    return polys


def _read_polygon_label(lb_file: str) -> np.ndarray:
    """Read a label file as a (N, 9) float32 polygon array (or (0, 9))."""
    if not os.path.exists(lb_file):
        return np.zeros((0, 9), dtype=np.float32)
    rows: List[List[float]] = []
    with open(lb_file, "r", encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) >= 9:
                rows.append([float(x) for x in p[:9]])
            elif len(p) == 5:
                c, cx, cy, w, h = (float(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4]))
                rows.append(_yolo_box_to_polygon(cx, cy, w, h))
    if rows:
        return np.array(rows, dtype=np.float32)
    return np.zeros((0, 9), dtype=np.float32)


def _polygon_to_yolo_line(poly: List[float]) -> str:
    vals = poly[:9]
    xs = vals[1::2]
    ys = vals[2::2]
    cx = (min(xs) + max(xs)) / 2.0
    cy = (min(ys) + max(ys)) / 2.0
    w = max(xs) - min(xs)
    h = max(ys) - min(ys)
    cx = min(max(cx, 0.0), 1.0)
    cy = min(max(cy, 0.0), 1.0)
    w = min(max(w, 0.0), 1.0)
    h = min(max(h, 0.0), 1.0)
    if w <= 0.0 or h <= 0.0:
        return ""
    return f"{int(vals[0])} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def _convert_polygon_label_file_to_yolo(lbl_path: str) -> None:
    """In-place convert a DOTA-polygon tile label file to standard YOLO."""
    if not os.path.exists(lbl_path):
        return
    out_lines: List[str] = []
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) >= 9:
                yolo = _polygon_to_yolo_line([float(x) for x in p[:9]])
                if yolo:
                    out_lines.append(yolo)
            elif len(p) == 5:
                out_lines.append(" ".join(p[:5]))  # already YOLO -> keep
    with open(lbl_path, "w", encoding="utf-8") as f:
        for ln in out_lines:
            f.write(ln + "\n")


## 12 · Phase 8 — Build tiled dataset (square crops) via `ultralytics.split_dota`

Optional dataset (`"tiled"` in `BUILD_DATASETS`). Uses `split_trainval` (train+val)
and `split_test` (test) on a temporary polygon-label bridge tree, then regenerates
`labels/test` (which `split_test` omits) and converts every tile label back to
standard YOLO.

In [13]:
def build_tiled_dataset(
    page_level_dir: str,
    tiled_dir: str,
    crop_size: int = 2048,
    gap: int = 200,
    rates: Tuple[float, ...] = (1.0,),
) -> Dict[str, Tuple[int, int]]:
    """Build Dataset 2 by tiling Dataset 1 with ultralytics' split_dota.

    Because split_dota expects DOTA polygon labels (and split_test emits no
    labels), we:
      1. build a temporary 'bridge' tree: images symlinked to Dataset 1, labels
         converted YOLO->polygon;
      2. run split_trainval (train+val) and split_test (test images) on it;
      3. generate labels/test with the official crop_and_save() so the test
         split has aligned labels too;
      4. convert every generated tile label polygon->YOLO;
      5. write data.yaml and remove the bridge.

    Returns: {split: (n_tiles, n_negative_tiles)}.
    """
    import cv2  # noqa: F401  (required by split_dota internals)
    from PIL import Image
    from ultralytics.data.split_dota import (
        crop_and_save,
        get_windows,
        get_window_obj,
        split_test,
        split_trainval,
    )
    from ultralytics.data.utils import exif_size, img2label_paths

    if os.path.exists(tiled_dir):
        shutil.rmtree(tiled_dir)
    os.makedirs(tiled_dir, exist_ok=True)

    bridge = tiled_dir + "_src"
    if os.path.exists(bridge):
        shutil.rmtree(bridge)

    # --- 1. bridge tree (symlinked images + polygon labels) ---------------- #
    for sp in SPLITS:
        os.makedirs(os.path.join(bridge, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(bridge, "labels", sp), exist_ok=True)
        src_lbl_dir = os.path.join(page_level_dir, "labels", sp)
        src_img_dir = os.path.join(page_level_dir, "images", sp)
        for lbl in glob.glob(os.path.join(src_lbl_dir, "*.txt")):
            stem = os.path.splitext(os.path.basename(lbl))[0]
            src_img = os.path.join(src_img_dir, stem + ".png")
            if os.path.exists(src_img):
                _link_or_copy(src_img, os.path.join(bridge, "images", sp, stem + ".png"))
            polys = _yolo_label_file_to_polygon(lbl)
            with open(os.path.join(bridge, "labels", sp, stem + ".txt"), "w", encoding="utf-8") as f:
                for poly in polys:
                    f.write(" ".join(f"{v:.6f}" for v in poly) + "\n")
            # empty label file -> 0 bytes (split_dota handles empty pages as background)

    # --- 2. tile train+val (images + polygon labels) ----------------------- #
    log.info("Tiling train+val (split_trainval) crop=%d gap=%d rates=%s", crop_size, gap, rates)
    split_trainval(
        data_root=bridge, save_dir=tiled_dir,
        crop_size=crop_size, gap=gap, rates=rates,
    )

    # --- 3. tile test images (split_test) ---------------------------------- #
    log.info("Tiling test (split_test) crop=%d gap=%d rates=%s", crop_size, gap, rates)
    split_test(
        data_root=bridge, save_dir=tiled_dir,
        crop_size=crop_size, gap=gap, rates=rates,
    )

    # --- 4. generate labels/test (split_test does not emit labels) --------- #
    log.info("Generating labels/test with crop_and_save (aligned with split_test naming)")
    crop_sizes = tuple(int(crop_size / r) for r in rates)
    gaps = tuple(int(gap / r) for r in rates)
    test_im_out = os.path.join(tiled_dir, "images", "test")
    test_lb_out = os.path.join(tiled_dir, "labels", "test")
    os.makedirs(test_lb_out, exist_ok=True)
    for im_file in sorted(glob.glob(os.path.join(bridge, "images", "test", "*"))):
        w, h = exif_size(Image.open(im_file))           # (width, height)
        lb_file = img2label_paths([im_file])[0]
        label = _read_polygon_label(lb_file)            # (N, 9) float32
        anno = dict(ori_size=(h, w), label=label, filepath=im_file)
        windows = get_windows(anno["ori_size"], crop_sizes=crop_sizes, gaps=gaps)
        window_objs = get_window_obj(anno, windows)
        crop_and_save(
            anno, windows, window_objs,
            str(test_im_out), str(test_lb_out),
            allow_background_images=True,
        )

    # --- 5. convert every tile label polygon -> YOLO ----------------------- #
    for sp in SPLITS:
        ld = os.path.join(tiled_dir, "labels", sp)
        if not os.path.isdir(ld):
            continue
        for lbl in glob.glob(os.path.join(ld, "*.txt")):
            _convert_polygon_label_file_to_yolo(lbl)

    # --- 6. data.yaml + cleanup -------------------------------------------- #
    _write_yaml(tiled_dir, DATA_YAML)
    shutil.rmtree(bridge, ignore_errors=True)
    log.info("Dataset 2 (tiled) written to %s", tiled_dir)

    # --- stats -------------------------------------------------------------- #
    stats: Dict[str, Tuple[int, int]] = {}
    for sp in SPLITS:
        im_dir = os.path.join(tiled_dir, "images", sp)
        tiles = glob.glob(os.path.join(im_dir, "*.jpg")) + glob.glob(os.path.join(im_dir, "*.png"))
        labelled = sum(
            1 for t in tiles if os.path.exists(
                os.path.join(tiled_dir, "labels", sp, os.path.splitext(os.path.basename(t))[0] + ".txt")
            )
        )
        n_tiles = len(tiles)
        n_neg = n_tiles - labelled
        stats[sp] = (n_tiles, n_neg)
        log.info("Tiled %s: %d tiles (%d negative/background)", sp, n_tiles, n_neg)
    return stats


## 13 · Phase 8b — Build strip dataset (horizontal bands, no x-cutting)

Default derived dataset (`"strips"` in `BUILD_DATASETS`). Each page is split into
`N_STRIPS` full-width horizontal strips with an **adaptive vertical overlap** so
no highlight is ever cut. Pure OpenCV — no ultralytics/shapely needed.

In [14]:
def build_strip_dataset(
    page_level_dir: str,
    strip_dir: str,
    n_strips: int = 4,
    gap_frac: float = 0.15,
    iof_thr: float = 0.99,
    allow_background: bool = True,
    img_quality: int = 95,
) -> Dict[str, Tuple[int, int]]:
    """Build a horizontal-strip dataset from the page-level dataset.

    Each page is split into `n_strips` full-width horizontal bands with an
    adaptive vertical overlap so that no highlight box is ever cut in half.
    Outputs JPG crops + standard YOLO labels (cls cx cy w h, normalised to the
    strip) and a data.yaml identical to the page-level one.

    Returns: {split: (n_strips_total, n_negative_strips)}.
    """
    import cv2

    if os.path.exists(strip_dir):
        shutil.rmtree(strip_dir)
    for sp in SPLITS:
        os.makedirs(os.path.join(strip_dir, "images", sp), exist_ok=True)
        os.makedirs(os.path.join(strip_dir, "labels", sp), exist_ok=True)

    def _read_yolo(lbl_path: str) -> List[List[float]]:
        out: List[List[float]] = []
        if not os.path.exists(lbl_path):
            return out
        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f:
                p = line.split()
                if len(p) >= 5:
                    out.append([float(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])])
        return out

    page_items: List[Tuple[str, str, str, str]] = []
    for sp in SPLITS:
        img_dir = os.path.join(page_level_dir, "images", sp)
        lbl_dir = os.path.join(page_level_dir, "labels", sp)
        for img_path in sorted(glob.glob(os.path.join(img_dir, "*.png"))):
            stem = os.path.splitext(os.path.basename(img_path))[0]
            page_items.append((sp, stem, img_path, os.path.join(lbl_dir, stem + ".txt")))

    stats: Dict[str, Tuple[int, int]] = {s: (0, 0) for s in SPLITS}
    for sp, stem, img_path, lbl_path in page_items:
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            log.warning("Could not read page image %s -> skipping", img_path)
            continue
        H, W = img.shape[:2]
        boxes = _read_yolo(lbl_path)  # [cls, cx, cy, w, h] normalised to page

        # --- strip geometry with adaptive overlap --------------------------- #
        max_bh_px = max((b[4] * H for b in boxes), default=0.0)
        gap = max(gap_frac * (H / n_strips), 1.5 * max_bh_px)
        gap = min(gap, H - 1)  # safety cap
        ch = (H + (n_strips - 1) * gap) / n_strips
        stride = ch - gap
        ys = [stride * i for i in range(n_strips)]
        if n_strips > 1 and ys[-1] + ch > H:
            ys[-1] = H - ch
        ys = [int(round(max(0.0, min(y, H - ch)))) for y in ys]
        ch = int(round(ch))

        # --- per strip: crop + assign boxes -------------------------------- #
        for i, y0 in enumerate(ys):
            y1 = min(H, y0 + ch)
            strip_img = img[y0:y1, :, :]
            sh, sw = strip_img.shape[:2]
            if sw == 0 or sh == 0:
                continue

            lines: List[str] = []
            for b in boxes:
                cls, cx, cy, w, h = b
                by1 = (cy - h / 2.0) * H
                by2 = (cy + h / 2.0) * H
                box_h = by2 - by1
                if box_h <= 0:
                    continue
                inter_h = max(0.0, min(by2, y1) - max(by1, y0))
                iof = inter_h / box_h
                if iof >= iof_thr:
                    # fully inside this strip -> keep whole, width unchanged
                    local_cx = min(max(cx, 0.0), 1.0)
                    local_w = min(max(w, 0.0), 1.0)
                    local_cy = ((cy * H) - y0) / sh
                    local_h = box_h / sh
                    local_cy = min(max(local_cy, 0.0), 1.0)
                    local_h = min(max(local_h, 0.0), 1.0)
                    if local_w > 0 and local_h > 0:
                        lines.append(f"{int(cls)} {local_cx:.6f} {local_cy:.6f} {local_w:.6f} {local_h:.6f}")
                elif box_h > ch and iof > 0.0:
                    # outlier taller than a strip -> clip to this strip (only the
                    # best-overlap strip will keep the largest iof)
                    cy1 = max(by1, y0); cy2 = min(by2, y1)
                    local_cx = min(max(cx, 0.0), 1.0)
                    local_w = min(max(w, 0.0), 1.0)
                    local_cy = ((cy1 + cy2) / 2.0 - y0) / sh
                    local_h = (cy2 - cy1) / sh
                    local_cy = min(max(local_cy, 0.0), 1.0)
                    local_h = min(max(local_h, 0.0), 1.0)
                    if local_w > 0 and local_h > 0 and iof >= 0.5:
                        lines.append(f"{int(cls)} {local_cx:.6f} {local_cy:.6f} {local_w:.6f} {local_h:.6f}")

            # best-overlap fallback for outlier boxes assigned to no strip yet
            if any(b[4] * H > ch for b in boxes):
                handled = set()
                for b in boxes:
                    cls, cx, cy, w, h = b
                    by1 = (cy - h / 2.0) * H; by2 = (cy + h / 2.0) * H
                    if (b[4] * H) <= ch:
                        continue
                    best_i = max(range(n_strips),
                                 key=lambda k: max(0.0, min(by2, ys[k] + ch) - max(by1, ys[k])))
                    if best_i != i:
                        continue
                    key = (round(cx, 6), round(cy, 6), round(w, 6), round(h, 6))
                    if key in handled:
                        continue
                    handled.add(key)
                    y0b, y1b = ys[best_i], min(H, ys[best_i] + ch)
                    cy1 = max(by1, y0b); cy2 = min(by2, y1b)
                    shb = y1b - y0b
                    local_cx = min(max(cx, 0.0), 1.0)
                    local_w = min(max(w, 0.0), 1.0)
                    local_cy = ((cy1 + cy2) / 2.0 - y0b) / shb
                    local_h = (cy2 - cy1) / shb
                    local_cy = min(max(local_cy, 0.0), 1.0)
                    local_h = min(max(local_h, 0.0), 1.0)
                    line = f"{int(cls)} {local_cx:.6f} {local_cy:.6f} {local_w:.6f} {local_h:.6f}"
                    if line not in lines and local_w > 0 and local_h > 0:
                        lines.append(line)

            has_label = len(lines) > 0
            if has_label or allow_background:
                out_img = os.path.join(strip_dir, "images", sp,
                                      f"{stem}__strip{i}__y{y0}__h{sh}.jpg")
                cv2.imwrite(out_img, strip_img, [cv2.IMWRITE_JPEG_QUALITY, img_quality])
                out_lbl = os.path.join(strip_dir, "labels", sp,
                                      f"{stem}__strip{i}__y{y0}__h{sh}.txt")
                with open(out_lbl, "w", encoding="utf-8") as f:
                    for ln in lines:
                        f.write(ln + "\n")

    # Count strips cleanly from disk (totals + negatives).
    for sp in SPLITS:
        im_dir = os.path.join(strip_dir, "images", sp)
        lb_dir = os.path.join(strip_dir, "labels", sp)
        imgs = glob.glob(os.path.join(im_dir, "*.jpg")) + glob.glob(os.path.join(im_dir, "*.png"))
        n_tot = len(imgs)
        n_neg = sum(1 for im in imgs if os.path.getsize(
            os.path.join(lb_dir, os.path.splitext(os.path.basename(im))[0] + ".txt")) == 0)
        stats[sp] = (n_tot, n_neg)
        log.info("Strips %s: %d strips (%d negative/background)", sp, n_tot, n_neg)

    _write_yaml(strip_dir, DATA_YAML)
    log.info("Dataset 2b (horizontal strips) written to %s", strip_dir)
    return stats


## 14 · Phase 9 — Zip datasets & final report

In [15]:
def zip_datasets(output_dir: str, names: List[str], keep_dirs: bool = False) -> List[str]:
    """Zip each requested dataset directory (e.g. 'dataset_page_level',
    'dataset_tiled', 'dataset_strips') into <name>.zip in output_dir.

    Unzipped directories are removed after being zipped (to keep peak disk
    usage low on Kaggle) unless keep_dirs=True.
    """
    zips: List[str] = []
    for name in names:
        d = os.path.join(output_dir, name)
        if not os.path.isdir(d):
            log.warning("Cannot zip %s: directory missing.", d)
            continue
        zp = shutil.make_archive(
            base_name=os.path.join(output_dir, name),
            format="zip", root_dir=output_dir, base_dir=name,
        )
        log.info("Created %s (%.1f MB)", zp, os.path.getsize(zp) / 1e6)
        zips.append(zp)
        if not keep_dirs and os.path.isdir(d):
            shutil.rmtree(d)
    return zips


# --------------------------------------------------------------------------- #
# Final report
# --------------------------------------------------------------------------- #
def print_report(stats: dict) -> None:
    line = "=" * 64
    print("\n" + line)
    print("DATASET BUILD REPORT")
    print(line)
    print(f"Processed PDFs           : {stats['n_pdfs']}")
    print(f"PDFs failed               : {stats['n_pdfs_failed']}")
    print(f"Rendered pages (total)    : {stats['n_pages_rendered']}")
    print(f"Selected pages            : {stats['n_selected']} "
          f"(pos={stats['n_pos']}, neg={stats['n_neg']})")
    print(f"Negative sample ratio     : {stats['neg_pct']:.1%}")
    print(f"Average skew angle        : {stats['avg_skew']:.3f} deg")
    print("-" * 64)
    print("Page-level dataset (pages per split):")
    for sp in SPLITS:
        print(f"  {sp:5s}: {stats['page_split'][sp]}")
    if stats.get("tile_split"):
        print("-" * 64)
        print("Tiled dataset (square tiles per split / negatives):")
        tot = totn = 0
        for sp in SPLITS:
            n, neg = stats["tile_split"][sp]; tot += n; totn += neg
            print(f"  {sp:5s}: {n} tiles ({neg} negative/background)")
        if tot:
            print(f"  tiled negative ratio    : {totn / tot:.1%}")
    if stats.get("strip_split"):
        print("-" * 64)
        print("Strip dataset (horizontal strips per split / negatives):")
        tot = totn = 0
        for sp in SPLITS:
            n, neg = stats["strip_split"][sp]; tot += n; totn += neg
            print(f"  {sp:5s}: {n} strips ({neg} negative/background)")
        if tot:
            print(f"  strip negative ratio    : {totn / tot:.1%}")
    print("-" * 64)
    print("Output archives:")
    for z in stats.get("zips", []):
        print(f"  {z}  ({os.path.getsize(z) / 1e6:.1f} MB)")
    print(line + "\n")


# --------------------------------------------------------------------------- #
# Main pipeline
# --------------------------------------------------------------------------- #


## 15 · Run the full pipeline

Runs phases 1-9 in order using the parameters from the Configuration cell.
Which datasets are built & zipped is controlled by `BUILD_DATASETS`.

In [16]:
random.seed(SEED)
np.random.seed(SEED)

ensure_deps()

build = list(dict.fromkeys(BUILD_DATASETS))  # dedupe, keep order
# page-level is always built (source for tiled/strips); zipped only if 'page' requested
zip_names: List[str] = []
if "page" in build:
    zip_names.append("dataset_page_level")
if "strips" in build:
    zip_names.append("dataset_strips")
if "tiled" in build:
    zip_names.append("dataset_tiled")

output_dir = os.path.abspath(OUTPUT_DIR)
os.makedirs(output_dir, exist_ok=True)

# Idempotency: wipe previous outputs.
for sub in ("dataset_page_level", "dataset_tiled", "dataset_strips", "_render_tmp",
            "dataset_tiled_src", "dataset_page_level.zip", "dataset_tiled.zip",
            "dataset_strips.zip"):
    p = os.path.join(output_dir, sub)
    if os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True)
    elif os.path.islink(p) or os.path.isfile(p):
        try:
            os.remove(p)
        except OSError:
            pass

page_level_dir = os.path.join(output_dir, "dataset_page_level")
tiled_dir = os.path.join(output_dir, "dataset_tiled")
strip_dir = os.path.join(output_dir, "dataset_strips")
tmp_dir = os.path.join(output_dir, "_render_tmp")
os.makedirs(tmp_dir, exist_ok=True)

# ---- 1. find pairs --------------------------------------------------- #
pairs = find_pdf_json_pairs(INPUT_DIR)
if not pairs:
    log.error("No PDF/JSON pairs found under %s", INPUT_DIR)
    raise SystemExit(1)

# ---- 2/3/4. render -> annotate (build full sample list) ---- #
all_samples: List[dict] = []
n_pages_rendered = 0
skew_angles: List[float] = []
n_pdfs_failed = 0

for idx, (pdf_path, json_path) in enumerate(pairs):
    stem_base = f"{idx:03d}_{sanitize_stem(os.path.splitext(os.path.basename(pdf_path))[0])}"
    try:
        rendered = render_pdf_pages(
            pdf_path, dpi=DPI, max_dim=MAX_DIM, out_dir=tmp_dir, stem=stem_base,
        )
    except Exception as e:
        log.error("Failed to open/render PDF (%s): %s", pdf_path, e)
        n_pdfs_failed += 1
        continue

    # annotations (pre-normalised in results.json)
    try:
        boxes_by_page = convert_annotations(json_path, img_size=None)
    except Exception as e:
        log.error("Malformed JSON %s: %s -> skipping PDF %s", json_path, e, pdf_path)
        n_pdfs_failed += 1
        continue

    for r in rendered:
        img_path = r["img_path"]
        page_num = r["page_num"]
        # deskew (overwrite in place)
        try:
            angle = deskew_image_file(img_path)
        except Exception as e:
            log.warning("Deskew failed for %s: %s (left as-is)", img_path, e)
            angle = 0.0
        skew_angles.append(angle)

        boxes = boxes_by_page.get(str(page_num), [])
        is_negative = (len(boxes) == 0)
        stem = os.path.splitext(os.path.basename(img_path))[0]
        all_samples.append(dict(
            pdf_key=stem_base, pdf_path=pdf_path, json_path=json_path,
            page_num=page_num, img_path=img_path, width=r["width"], height=r["height"],
            boxes=boxes, is_negative=is_negative, stem=stem, skew_angle=angle,
        ))
        n_pages_rendered += 1

    log.info("Processed [%d/%d] %s  pages=%d  (skew avg=%.2f deg)",
             idx + 1, len(pairs), os.path.basename(pdf_path),
             len(rendered),
             (sum(skew_angles[-len(rendered):]) / max(1, len(rendered))))

log.info("Rendered+deskewed %d pages from %d PDFs (%d PDFs failed).",
         n_pages_rendered, len(pairs), n_pdfs_failed)

if not all_samples:
    log.error("No pages produced. Aborting.")
    raise SystemExit(1)

# ---- 5. balance negatives ------------------------------------------- #
selected = balance_negatives(all_samples, target_pct=NEG_PCT)
n_pos = sum(1 for s in selected if not s["is_negative"])
n_neg = sum(1 for s in selected if s["is_negative"])
neg_pct = n_neg / len(selected) if selected else 0.0

# ---- 6. split (no leakage) ------------------------------------------ #
splits = split_no_leakage(selected, ratios=(0.7, 0.15, 0.15))

# ---- 7. Dataset 1 (page-level) -- always built as the source --------- #
page_counts = build_page_level_dataset(splits, page_level_dir)
shutil.rmtree(tmp_dir, ignore_errors=True)

# ---- 8. Derived datasets -------------------------------------------- #
strip_stats: Dict[str, Tuple[int, int]] = {s: (0, 0) for s in SPLITS}
if "strips" in build:
    try:
        strip_stats = build_strip_dataset(
            page_level_dir, strip_dir,
            n_strips=N_STRIPS, gap_frac=STRIP_GAP_FRAC,
            iof_thr=0.99, allow_background=True,
        )
    except Exception as e:
        log.error("Strip dataset build failed: %s\n%s", e, traceback.format_exc())

tile_stats: Dict[str, Tuple[int, int]] = {s: (0, 0) for s in SPLITS}
if "tiled" in build:
    try:
        tile_stats = build_tiled_dataset(
            page_level_dir, tiled_dir,
            crop_size=CROP_SIZE, gap=GAP, rates=(1.0,),
        )
    except Exception as e:
        log.error("Tiled dataset build failed: %s\n%s", e, traceback.format_exc())

# If the page-level dataset itself is not requested, free its disk now that
# the derived datasets have been built from it.
if "page" not in build and os.path.isdir(page_level_dir):
    shutil.rmtree(page_level_dir, ignore_errors=True)

# ---- 9. zip requested datasets -------------------------------------- #
zips: List[str] = []
try:
    zips = zip_datasets(output_dir, zip_names, keep_dirs=KEEP_DIRS)
except Exception as e:
    log.error("Zipping failed: %s\n%s", e, traceback.format_exc())

# ---- report --------------------------------------------------------- #
avg_skew = float(np.mean(skew_angles)) if skew_angles else 0.0
print_report(dict(
    n_pdfs=len(pairs), n_pdfs_failed=n_pdfs_failed,
    n_pages_rendered=n_pages_rendered,
    n_selected=len(selected), n_pos=n_pos, n_neg=n_neg, neg_pct=neg_pct,
    avg_skew=avg_skew,
    page_split=page_counts,
    tile_split=tile_stats if "tiled" in build else None,
    strip_split=strip_stats if "strips" in build else None,
    zips=zips,
))
pass  # done


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


16:31:26 [INFO] Found 103 PDF/JSON pair(s); skipped 0 PDF(s) without annotations.
16:31:27 [INFO] Processed [1/103] 1)I livelli di organizzazione della materia vivente 0.pdf  pages=4  (skew avg=0.00 deg)
16:31:29 [INFO] Processed [2/103] 2)La forma delle cellula non è univoca.pdf  pages=6  (skew avg=0.00 deg)
16:31:31 [INFO] Processed [3/103] 2_Multi_agent_LLMs.pdf  pages=8  (skew avg=0.00 deg)
16:31:34 [INFO] Processed [4/103] 2_Multi_agent_sequential_LLMs.pdf  pages=9  (skew avg=0.00 deg)
16:31:41 [INFO] Processed [5/103] 2_Survey_of_LLM-based_Text-to-SQL.pdf  pages=20  (skew avg=0.00 deg)
16:31:43 [INFO] Processed [6/103] 3)La cellula e la sua membrana plasmatica 1.pdf  pages=6  (skew avg=0.00 deg)
16:31:44 [INFO] Processed [7/103] 4)LA MEMBRANA PLASMATICA.pdf  pages=4  (skew avg=0.00 deg)
16:32:43 [INFO] Processed [8/103] ANATOMIA.pdf  pages=76  (skew avg=0.00 deg)
16:32:49 [INFO] Processed [9/103] Animali da laboratorio_250226_144342.pdf  pages=22  (skew avg=0.00 deg)
16:33:00 [I


DATASET BUILD REPORT
Processed PDFs           : 103
PDFs failed               : 0
Rendered pages (total)    : 2438
Selected pages            : 2197 (pos=1933, neg=264)
Negative sample ratio     : 12.0%
Average skew angle        : 0.000 deg
----------------------------------------------------------------
Page-level dataset (pages per split):
  train: 1598
  val  : 337
  test : 262
----------------------------------------------------------------
Output archives:
  /kaggle/working/dataset_page_level.zip  (6206.4 MB)



## 16 · Inspect outputs

After the pipeline finishes, the requested archives are available at
`/kaggle/working/`:

- `dataset_page_level.zip`  (if `"page"` in BUILD_DATASETS)
- `dataset_strips.zip`      (if `"strips"` in BUILD_DATASETS)
- `dataset_tiled.zip`       (if `"tiled"` in BUILD_DATASETS)


In [17]:
import os
for z in ("/kaggle/working/dataset_page_level.zip",
          "/kaggle/working/dataset_strips.zip",
          "/kaggle/working/dataset_tiled.zip"):
    if os.path.exists(z):
        print(f"{z}  ->  {os.path.getsize(z)/1e6:.1f} MB")
    else:
        print(f"{z}  ->  not built")


/kaggle/working/dataset_page_level.zip  ->  6206.4 MB
/kaggle/working/dataset_strips.zip  ->  not built
/kaggle/working/dataset_tiled.zip  ->  not built
